
# NASA Asteroids: przygotowanie danych i wizualizacja PCA

Pierwsza część notebooka pokazuje redukcję wymiarów dla danych opisujących planetoidy bliskie Ziemi.

Cel:

```text
kilkanaście parametrów fizycznych i orbitalnych
→ wybór cech numerycznych
→ skalowanie
→ PCA do 2 wymiarów
→ wizualizacja podobieństwa obiektów
```

> Etykieta `Hazardous` oznacza obiekt sklasyfikowany jako potencjalnie niebezpieczny. Nie oznacza to automatycznie, że planetoida znajduje się na kursie kolizyjnym z Ziemią.

## Dane

Umieść plik Kaggle/CSV lub Parquet w katalogu `data/` i ustaw `DATA_PATH`.

Notebook rozpoznaje popularne nazwy kolumn ze zbioru **NASA: Asteroids Classification**. Jeżeli pliku nie ma, tworzy syntetyczny zbiór demonstracyjny wyłącznie do sprawdzenia kodu.


In [ ]:

# W środowisku lokalnym odkomentuj w razie potrzeby:
# %pip install -U polars pyarrow scikit-learn matplotlib

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("Polars:", pl.__version__)


In [ ]:

DATA_PATH = Path("data/nasa_asteroids.csv")
CREATE_DEMO_IF_MISSING = True

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



## 1. Zbiór demonstracyjny

Fallback zawiera przykładowe cechy przypominające parametry fizyczne i orbitalne oraz kilka brakujących wartości. W tej części PCA użyjemy tylko kompletnych rekordów. W kolejnej części notebooka zastąpimy to imputacją.


In [ ]:

def create_demo_asteroids(path: Path, n_rows: int = 600) -> None:
    rng = np.random.default_rng(42)

    absolute_magnitude = rng.normal(22.0, 2.3, n_rows)
    estimated_diameter_min = np.exp(rng.normal(-2.0, 0.65, n_rows))
    estimated_diameter_max = estimated_diameter_min * rng.uniform(1.5, 2.4, n_rows)
    relative_velocity = np.exp(rng.normal(9.6, 0.35, n_rows))
    miss_distance_km = np.exp(rng.normal(15.2, 0.8, n_rows))
    minimum_orbit_intersection = np.exp(rng.normal(-3.0, 0.9, n_rows))
    eccentricity = np.clip(rng.beta(2.0, 3.5, n_rows), 0.01, 0.98)
    semi_major_axis = rng.uniform(0.7, 4.5, n_rows)
    inclination = np.clip(rng.gamma(2.0, 6.0, n_rows), 0, 70)
    orbital_period = np.exp(rng.normal(6.3, 0.45, n_rows))
    perihelion_distance = np.clip(
        semi_major_axis * (1 - eccentricity),
        0.05,
        None,
    )
    aphelion_distance = semi_major_axis * (1 + eccentricity)

    risk_score = (
        -0.85 * absolute_magnitude
        - 2.0 * np.log1p(minimum_orbit_intersection)
        + 0.15 * np.log1p(relative_velocity)
        + rng.normal(0, 0.9, n_rows)
    )
    hazardous = risk_score > np.quantile(risk_score, 0.82)

    demo = pl.DataFrame(
        {
            "Neo Reference ID": np.arange(1_000_000, 1_000_000 + n_rows),
            "Name": [f"Demo Asteroid {i}" for i in range(n_rows)],
            "Absolute Magnitude": absolute_magnitude,
            "Est Dia in KM(min)": estimated_diameter_min,
            "Est Dia in KM(max)": estimated_diameter_max,
            "Relative Velocity km per hr": relative_velocity,
            "Miss Dist.(kilometers)": miss_distance_km,
            "Minimum Orbit Intersection": minimum_orbit_intersection,
            "Eccentricity": eccentricity,
            "Semi Major Axis": semi_major_axis,
            "Inclination": inclination,
            "Orbital Period": orbital_period,
            "Perihelion Distance": perihelion_distance,
            "Aphelion Dist": aphelion_distance,
            "Hazardous": hazardous,
        }
    )

    # Kilka braków do późniejszego segmentu o imputacji.
    missing_indices = rng.choice(n_rows, size=max(1, n_rows // 25), replace=False)
    demo = demo.with_row_index("_row_id").with_columns(
        pl.when(pl.col("_row_id").is_in(missing_indices[: len(missing_indices)//2]))
        .then(None)
        .otherwise(pl.col("Absolute Magnitude"))
        .alias("Absolute Magnitude"),
        pl.when(pl.col("_row_id").is_in(missing_indices[len(missing_indices)//2:]))
        .then(None)
        .otherwise(pl.col("Minimum Orbit Intersection"))
        .alias("Minimum Orbit Intersection"),
    ).drop("_row_id")

    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix.lower() == ".parquet":
        demo.write_parquet(path)
    else:
        demo.write_csv(path)


if not DATA_PATH.exists():
    if not CREATE_DEMO_IF_MISSING:
        raise FileNotFoundError(
            f"Nie znaleziono {DATA_PATH}. Ustaw ścieżkę do CSV lub Parquet."
        )

    create_demo_asteroids(DATA_PATH)
    print(
        f"UWAGA: utworzono syntetyczny zbiór demonstracyjny: {DATA_PATH}. "
        "Do prezentacji użyj właściwego zbioru NASA Asteroids."
    )



## 2. Wczytanie danych

Dla CSV i Parquet korzystamy z Lazy API Polars.


In [ ]:

def scan_dataset(path: Path) -> pl.LazyFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pl.scan_csv(
            path,
            infer_schema_length=10_000,
            try_parse_dates=False,
        )

    if suffix == ".parquet":
        return pl.scan_parquet(path)

    raise ValueError("Obsługiwane formaty: CSV i Parquet.")


raw_lf = scan_dataset(DATA_PATH)
raw_lf.collect_schema()



## 3. Ujednolicenie nazw kolumn

Różne kopie zbioru mogą nieznacznie różnić się zapisem nazw. Tworzymy nazwy w formacie `snake_case`.


In [ ]:

def to_snake_case(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


source_columns = raw_lf.collect_schema().names()
rename_map = {column: to_snake_case(column) for column in source_columns}

asteroids_lf = raw_lf.rename(rename_map)
asteroids_lf.collect_schema()



## 4. Wybór cech numerycznych

Nie przekazujemy do PCA:

- technicznych identyfikatorów,
- nazw obiektów,
- targetu `hazardous`,
- tekstowych pól opisowych,
- kilku redundantnych przeliczeń tej samej wielkości w różnych jednostkach.

Poniższa lista obejmuje popularne kolumny ze zbioru NASA Asteroids. Notebook wykorzysta te, które faktycznie znajdują się w pliku.


In [ ]:

PREFERRED_NUMERIC_FEATURES = [
    "absolute_magnitude",
    "est_dia_in_km_min",
    "est_dia_in_km_max",
    "relative_velocity_km_per_hr",
    "miss_dist_kilometers",
    "minimum_orbit_intersection",
    "jupiter_tisserand_invariant",
    "orbit_uncertainity",
    "eccentricity",
    "semi_major_axis",
    "inclination",
    "asc_node_longitude",
    "orbital_period",
    "perihelion_distance",
    "perihelion_arg",
    "aphelion_dist",
    "mean_anomaly",
    "mean_motion",
]

schema = asteroids_lf.collect_schema()
available_columns = set(schema.names())

selected_features = [
    column
    for column in PREFERRED_NUMERIC_FEATURES
    if column in available_columns
]

if len(selected_features) < 3:
    raise ValueError(
        "Znaleziono zbyt mało rozpoznanych cech numerycznych. "
        f"Dostępne kolumny: {schema.names()}"
    )

print("Wybrane cechy:")
for feature in selected_features:
    print("-", feature)



## 5. Konwersja typów i analiza braków

W tej części nie omawiamy jeszcze imputacji. Sprawdzamy jedynie kompletność cech.


In [ ]:

typed_lf = asteroids_lf.with_columns(
    [
        pl.col(column).cast(pl.Float64, strict=False)
        for column in selected_features
    ]
)

missing_report = typed_lf.select(
    [
        pl.col(column).is_null().sum().alias(column)
        for column in selected_features
    ]
).collect()

missing_report



## 6. Tymczasowy zbiór kompletnych rekordów

Na potrzeby pierwszego przykładu PCA wybieramy tylko rekordy kompletne dla analizowanych cech.

To celowe uproszczenie. W kolejnym segmencie pokażemy, dlaczego usuwanie rekordów nie zawsze jest dobrym rozwiązaniem i zastosujemy `KNNImputer`.


In [ ]:

complete_cases = (
    typed_lf
    .select(selected_features)
    .drop_nulls()
    .collect()
)

print(f"Kompletne rekordy: {complete_cases.height:,}")
print(f"Liczba cech: {len(selected_features)}")

if complete_cases.height < 10:
    raise ValueError(
        "Po usunięciu braków pozostało zbyt mało rekordów do PCA."
    )



## 7. Skalowanie

PCA maksymalizuje wariancję. Bez skalowania cechy zapisane w dużych jednostkach, np. odległość w kilometrach, mogłyby zdominować komponenty.


In [ ]:

X_asteroids = complete_cases.to_numpy()

if not np.isfinite(X_asteroids).all():
    raise ValueError("Dane zawierają wartości nieskończone.")

asteroid_scaler = StandardScaler()
X_asteroids_scaled = asteroid_scaler.fit_transform(X_asteroids)

pl.DataFrame(
    {
        "feature": selected_features,
        "mean_after_scaling": X_asteroids_scaled.mean(axis=0),
        "std_after_scaling": X_asteroids_scaled.std(axis=0),
    }
)



## 8. PCA do dwóch wymiarów


In [ ]:

asteroid_pca_2d = PCA(n_components=2)
X_asteroids_pca = asteroid_pca_2d.fit_transform(X_asteroids_scaled)

asteroid_variance = pl.DataFrame(
    {
        "component": ["PC1", "PC2"],
        "explained_variance_ratio": asteroid_pca_2d.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(
            asteroid_pca_2d.explained_variance_ratio_
        ),
    }
)

asteroid_variance


In [ ]:

plt.figure(figsize=(8.5, 6))
plt.scatter(
    X_asteroids_pca[:, 0],
    X_asteroids_pca[:, 1],
    s=18,
    alpha=0.6,
)
plt.title("Planetoidy w przestrzeni dwóch komponentów PCA")
plt.xlabel(
    f"PC1 ({asteroid_pca_2d.explained_variance_ratio_[0]:.1%} wariancji)"
)
plt.ylabel(
    f"PC2 ({asteroid_pca_2d.explained_variance_ratio_[1]:.1%} wariancji)"
)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



## 9. Interpretacja komponentów

Komponenty nie odpowiadają jednej konkretnej kolumnie. Sprawdzamy więc loadings, aby zobaczyć, które cechy mają największy udział w PC1 i PC2.


In [ ]:

asteroid_loadings = pl.DataFrame(
    {
        "feature": selected_features,
        "PC1_loading": asteroid_pca_2d.components_[0],
        "PC2_loading": asteroid_pca_2d.components_[1],
    }
).with_columns(
    (
        pl.col("PC1_loading").abs()
        + pl.col("PC2_loading").abs()
    ).alias("combined_absolute_loading")
).sort(
    "combined_absolute_loading",
    descending=True,
)

asteroid_loadings



## 10. Ile komponentów potrzeba do zachowania większej części informacji?

Dwa komponenty są wygodne do wizualizacji, ale nie muszą zachowywać wystarczającej części wariancji do dalszego modelowania.

Dopasowujemy PCA bez ograniczenia liczby komponentów i sprawdzamy wariancję skumulowaną.


In [ ]:

asteroid_pca_full = PCA()
asteroid_pca_full.fit(X_asteroids_scaled)

cumulative_variance = np.cumsum(
    asteroid_pca_full.explained_variance_ratio_
)

components_for_80 = int(np.argmax(cumulative_variance >= 0.80) + 1)
components_for_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)

print("Komponenty potrzebne do 80% wariancji:", components_for_80)
print("Komponenty potrzebne do 90% wariancji:", components_for_90)


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o",
)
plt.axhline(0.80, linestyle="--")
plt.axhline(0.90, linestyle="--")
plt.title("Skumulowana wariancja wyjaśniona przez PCA")
plt.xlabel("Liczba komponentów")
plt.ylabel("Cumulative explained variance")
plt.xticks(range(1, len(cumulative_variance) + 1))
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



## Podsumowanie części PCA

```text
parametry asteroid
→ kompletne rekordy
→ StandardScaler
→ PCA do 2 wymiarów
→ wykres podobieństwa obiektów
```

Najważniejsze wnioski:

1. Jeden wymiar to jedna cecha opisująca rekord.
2. PCA tworzy nowe komponenty będące kombinacjami cech.
3. Dwa komponenty służą przede wszystkim wizualizacji.
4. `explained variance` mówi, ile informacji o zmienności danych zachowuje projekcja.
5. PCA wymaga świadomego skalowania.
6. PCA redukuje wymiary, ale nie wybiera oryginalnych kolumn.

W następnej części zastąpimy usuwanie niekompletnych rekordów imputacją oraz wykorzystamy dane do wykrywania anomalii.



---

## Część 2. Brakujące dane i wykrywanie anomalii

Przepływ:

```text
dane z brakami
→ porównanie SimpleImputer i KNNImputer
→ dopasowanie preprocessingu na zbiorze treningowym
→ przygotowanie kompletnej macierzy
→ Isolation Forest / LOF / K-Means / DBSCAN
→ porównanie wykrytych anomalii
```

W tej części rozróżniamy:

- **brak wartości** — nie znamy pomiaru,
- **anomalię** — wartość istnieje, ale rekord odbiega od typowego wzorca,
- **błąd danych** — rekord może być niepoprawny, ale algorytm sam tego nie rozstrzyga.



## 11. Przygotowanie macierzy z brakami

Korzystamy ze wszystkich rekordów, również tych zawierających brakujące wartości.

Aby porównać metody imputacji, dodatkowo ukrywamy niewielką część znanych wartości. Dzięki temu po imputacji możemy porównać wynik z rzeczywistą wartością.

Sztuczne braki są tworzone wyłącznie w kopii danych na potrzeby demonstracji.


In [ ]:

from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.pipeline import Pipeline

asteroid_features_df = typed_lf.select(selected_features).collect()
X_original = asteroid_features_df.to_numpy()

if X_original.shape[0] < 30:
    raise ValueError(
        "Do demonstracji potrzebujemy co najmniej 30 rekordów."
    )

real_missing_count = int(np.isnan(X_original).sum())
print("Rzeczywiste braki:", real_missing_count)
print("Kształt macierzy:", X_original.shape)


In [ ]:

rng = np.random.default_rng(42)

X_demo_missing = X_original.copy()
observed_positions = np.argwhere(np.isfinite(X_original))

SIMULATED_MISSING_RATE = 0.03
n_to_mask = max(
    1,
    int(len(observed_positions) * SIMULATED_MISSING_RATE),
)

chosen_positions = observed_positions[
    rng.choice(
        len(observed_positions),
        size=n_to_mask,
        replace=False,
    )
]

synthetic_missing_mask = np.zeros_like(
    X_original,
    dtype=bool,
)
synthetic_missing_mask[
    chosen_positions[:, 0],
    chosen_positions[:, 1],
] = True

X_demo_missing[synthetic_missing_mask] = np.nan

print("Dodane braki demonstracyjne:", n_to_mask)
print(
    "Łączna liczba braków w kopii:",
    int(np.isnan(X_demo_missing).sum()),
)



## 12. SimpleImputer

`SimpleImputer` uzupełnia każdą kolumnę niezależnie.

Dla zmiennych numerycznych typowe strategie to:

- `mean` — średnia,
- `median` — mediana,
- `constant` — stała wartość.

Dla zmiennych kategorycznych często wykorzystuje się:

- `most_frequent` — najczęstsza wartość,
- `constant`, np. `"unknown"`.

Mediana jest zwykle mniej wrażliwa na skrajne wartości niż średnia.


In [ ]:

row_indices = np.arange(X_demo_missing.shape[0])

train_indices, test_indices = train_test_split(
    row_indices,
    test_size=0.25,
    random_state=42,
)

X_train_missing = X_demo_missing[train_indices]
X_test_missing = X_demo_missing[test_indices]

simple_imputer = SimpleImputer(
    strategy="median",
    add_indicator=False,
)

simple_imputer.fit(X_train_missing)

X_train_simple = simple_imputer.transform(
    X_train_missing
)
X_test_simple = simple_imputer.transform(
    X_test_missing
)

print("Braki po imputacji train:", np.isnan(X_train_simple).sum())
print("Braki po imputacji test:", np.isnan(X_test_simple).sum())



## 13. KNNImputer

`KNNImputer` szuka rekordów podobnych według dostępnych cech i wykorzystuje ich wartości do uzupełnienia braku.

Ponieważ metoda opiera się na odległościach, wcześniej standaryzujemy kolumny. `StandardScaler` jest dopasowywany wyłącznie na zbiorze treningowym, a następnie stosowany do zbioru testowego.

Wagi `distance` powodują, że bliżsi sąsiedzi mają większy wpływ na wynik.


In [ ]:

knn_scaler = StandardScaler()
knn_scaler.fit(X_train_missing)

X_train_scaled_with_nan = knn_scaler.transform(
    X_train_missing
)
X_test_scaled_with_nan = knn_scaler.transform(
    X_test_missing
)

knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="distance",
)

knn_imputer.fit(X_train_scaled_with_nan)

X_train_knn_scaled = knn_imputer.transform(
    X_train_scaled_with_nan
)
X_test_knn_scaled = knn_imputer.transform(
    X_test_scaled_with_nan
)

# Powrót do oryginalnych jednostek tylko do porównania jakości imputacji.
X_train_knn = knn_scaler.inverse_transform(
    X_train_knn_scaled
)
X_test_knn = knn_scaler.inverse_transform(
    X_test_knn_scaled
)

print("Braki po KNN train:", np.isnan(X_train_knn).sum())
print("Braki po KNN test:", np.isnan(X_test_knn).sum())



## 14. Porównanie imputacji na sztucznie ukrytych wartościach

Porównujemy tylko te komórki w zbiorze testowym, które:

1. pierwotnie miały znaną wartość,
2. zostały przez nas celowo ukryte.

Ze względu na różne jednostki cech obliczamy błąd znormalizowany przez odchylenie standardowe danej kolumny.


In [ ]:

test_synthetic_mask = synthetic_missing_mask[test_indices]
X_test_truth = X_original[test_indices]

feature_scale = np.nanstd(
    X_original[train_indices],
    axis=0,
)
feature_scale = np.where(
    feature_scale > 0,
    feature_scale,
    1.0,
)

def normalized_imputation_mae(
    imputed: np.ndarray,
    truth: np.ndarray,
    mask: np.ndarray,
    scale: np.ndarray,
) -> float:
    row_idx, col_idx = np.where(mask)

    if len(row_idx) == 0:
        return float("nan")

    normalized_errors = (
        np.abs(
            imputed[row_idx, col_idx]
            - truth[row_idx, col_idx]
        )
        / scale[col_idx]
    )
    return float(np.mean(normalized_errors))


simple_error = normalized_imputation_mae(
    X_test_simple,
    X_test_truth,
    test_synthetic_mask,
    feature_scale,
)

knn_error = normalized_imputation_mae(
    X_test_knn,
    X_test_truth,
    test_synthetic_mask,
    feature_scale,
)

imputation_comparison = pl.DataFrame(
    {
        "method": [
            "SimpleImputer — median",
            "KNNImputer — 5 neighbors",
        ],
        "normalized_mae": [
            simple_error,
            knn_error,
        ],
    }
).sort("normalized_mae")

imputation_comparison



Niższy błąd w tej pojedynczej symulacji nie dowodzi, że dana metoda zawsze będzie najlepsza.

Wynik zależy między innymi od:

- mechanizmu powstawania braków,
- liczby rekordów,
- zależności pomiędzy cechami,
- skalowania,
- liczby sąsiadów,
- obecności anomalii.

Prosta mediana może być równie dobra albo lepsza od bardziej złożonej metody.



## 15. Opcjonalnie: IterativeImputer

`IterativeImputer` traktuje kolejno każdą niekompletną kolumnę jako zmienną przewidywaną na podstawie pozostałych cech.

W scikit-learn jest nadal funkcją eksperymentalną i wymaga jawnego włączenia:

```python
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(
    random_state=42,
    max_iter=10,
)
```

Nie używamy go dalej w tym krótkim przykładzie, aby nie zwiększać czasu obliczeń.



## 16. Dlaczego imputer dopasowujemy tylko na danych treningowych?

Imputer uczy się informacji ze zbioru:

- median,
- średnich,
- układu najbliższych sąsiadów,
- zależności pomiędzy kolumnami.

Jeżeli dopasujemy go na całych danych przed oceną modelu, informacje ze zbioru testowego wpłyną na preprocessing. Jest to data leakage.

Poprawny schemat:

```text
train → fit imputer/scaler
test  → tylko transform
```

W następnym punkcie połączymy te operacje z modelem w `Pipeline`.



# Część B. Wykrywanie anomalii

Dla historycznej, eksploracyjnej analizy przygotowujemy kompletną i przeskalowaną macierz wszystkich asteroid.

Tutaj preprocessing jest dopasowany do badanego zbioru referencyjnego. W systemie obsługującym nowe obiekty należałoby zachować imputer, scaler i detektor dopasowane na danych referencyjnych, a nowe dane wyłącznie transformować i oceniać.


In [ ]:

anomaly_preprocessor = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "imputer",
            KNNImputer(
                n_neighbors=5,
                weights="distance",
            ),
        ),
    ]
)

X_anomaly = anomaly_preprocessor.fit_transform(
    X_original
)

if not np.isfinite(X_anomaly).all():
    raise ValueError(
        "Po preprocessingu pozostały wartości NaN lub nieskończone."
    )

print("Macierz do anomaly detection:", X_anomaly.shape)



## 17. Isolation Forest

Isolation Forest tworzy wiele losowych drzew i próbuje izolować obserwacje przez losowe podziały.

Nietypowy rekord zwykle wymaga mniejszej liczby podziałów, ponieważ znajduje się daleko od gęstej części danych.

Parametr `contamination` określa oczekiwany udział rekordów, które zostaną oznaczone jako anomalie. Nie jest to wartość, którą powinniśmy ustawiać bez uzasadnienia biznesowego.


In [ ]:

EXPECTED_ANOMALY_SHARE = 0.03

isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=EXPECTED_ANOMALY_SHARE,
    random_state=42,
)

isolation_labels = isolation_forest.fit_predict(
    X_anomaly
)

# Wyższy wynik = bardziej nietypowy rekord.
isolation_anomaly_score = -isolation_forest.score_samples(
    X_anomaly
)

print(
    "Isolation Forest — anomalie:",
    int(np.sum(isolation_labels == -1)),
)



## 18. Local Outlier Factor

LOF porównuje lokalną gęstość rekordu z gęstością jego najbliższych sąsiadów.

Rekord może zostać uznany za nietypowy, jeżeli znajduje się w znacznie rzadszym obszarze niż jego otoczenie.

LOF może więc znajdować lokalne anomalie, które nie muszą być ekstremalne względem całego zbioru.


In [ ]:

lof_neighbors = min(
    20,
    max(2, X_anomaly.shape[0] - 1),
)

lof = LocalOutlierFactor(
    n_neighbors=lof_neighbors,
    contamination=EXPECTED_ANOMALY_SHARE,
)

lof_labels = lof.fit_predict(X_anomaly)

# negative_outlier_factor_ jest bardziej ujemny dla anomalii.
lof_anomaly_score = -lof.negative_outlier_factor_

print(
    "LOF — anomalie:",
    int(np.sum(lof_labels == -1)),
)



## 19. Odległość od centroidu K-Means

To podejście mierzy nietypowość względem najbliższego segmentu.

Po dopasowaniu K-Means obliczamy odległość każdego obiektu od przypisanego centroidu. Za potencjalne anomalie uznajemy rekordy powyżej wybranego kwantyla odległości.

Jest to heurystyka podobna do tej zastosowanej wcześniej dla klientów Online Retail.


In [ ]:

ANOMALY_K = min(
    5,
    max(2, X_anomaly.shape[0] // 20),
)

asteroid_kmeans = KMeans(
    n_clusters=ANOMALY_K,
    n_init=20,
    random_state=42,
)

asteroid_cluster_labels = asteroid_kmeans.fit_predict(
    X_anomaly
)

asteroid_centroid_distances = asteroid_kmeans.transform(
    X_anomaly
)

assigned_centroid_distance = asteroid_centroid_distances[
    np.arange(len(asteroid_cluster_labels)),
    asteroid_cluster_labels,
]

centroid_threshold = float(
    np.quantile(
        assigned_centroid_distance,
        1 - EXPECTED_ANOMALY_SHARE,
    )
)

centroid_anomaly_labels = np.where(
    assigned_centroid_distance >= centroid_threshold,
    -1,
    1,
)

print(
    "Odległość od centroidu — anomalie:",
    int(np.sum(centroid_anomaly_labels == -1)),
)



## 20. DBSCAN i rekordy `-1`

DBSCAN nie jest klasycznym detektorem anomalii, ale rekordy nieprzypisane do żadnego odpowiednio gęstego obszaru otrzymują etykietę `-1`.

Dobór `eps` ma bardzo duży wpływ na udział szumu. Poniżej stosujemy prostą heurystykę opartą na odległości do dziesiątego sąsiada.


In [ ]:

dbscan_min_samples = min(
    10,
    max(3, X_anomaly.shape[0] // 50),
)

neighbors = NearestNeighbors(
    n_neighbors=dbscan_min_samples,
)
neighbors.fit(X_anomaly)

neighbor_distances, _ = neighbors.kneighbors(
    X_anomaly
)

k_distance = neighbor_distances[:, -1]
dbscan_eps = float(np.quantile(k_distance, 0.95))

asteroid_dbscan = DBSCAN(
    eps=dbscan_eps,
    min_samples=dbscan_min_samples,
)

dbscan_labels = asteroid_dbscan.fit_predict(
    X_anomaly
)

print("DBSCAN eps:", round(dbscan_eps, 3))
print(
    "DBSCAN — noise:",
    int(np.sum(dbscan_labels == -1)),
)
print(
    "DBSCAN — zwykłe klastry:",
    len(set(dbscan_labels) - {-1}),
)



## 21. Porównanie wyników

Nie oczekujemy, że wszystkie metody wskażą dokładnie te same rekordy.

- Isolation Forest szuka obserwacji łatwych do odizolowania globalnie.
- LOF porównuje lokalną gęstość.
- K-Means mierzy odległość od centrum najbliższej grupy.
- DBSCAN wskazuje rekordy poza gęstymi obszarami.

Zgodność kilku metod może zwiększać priorytet ręcznej weryfikacji, ale nie dowodzi automatycznie, że rekord jest błędem.


In [ ]:

anomaly_flags = pl.DataFrame(
    {
        "isolation_forest_anomaly": isolation_labels == -1,
        "lof_anomaly": lof_labels == -1,
        "centroid_anomaly": centroid_anomaly_labels == -1,
        "dbscan_noise": dbscan_labels == -1,
        "isolation_score": isolation_anomaly_score,
        "lof_score": lof_anomaly_score,
        "centroid_distance": assigned_centroid_distance,
        "dbscan_cluster_id": dbscan_labels,
    }
).with_columns(
    pl.sum_horizontal(
        pl.col(
            "isolation_forest_anomaly",
            "lof_anomaly",
            "centroid_anomaly",
            "dbscan_noise",
        ).cast(pl.Int8)
    ).alias("number_of_methods_flagging")
)

method_comparison = anomaly_flags.select(
    pl.len().alias("records"),
    pl.col("isolation_forest_anomaly").sum().alias(
        "isolation_forest"
    ),
    pl.col("lof_anomaly").sum().alias("lof"),
    pl.col("centroid_anomaly").sum().alias(
        "centroid_distance"
    ),
    pl.col("dbscan_noise").sum().alias("dbscan_noise"),
    (
        pl.col("number_of_methods_flagging") >= 2
    ).sum().alias("flagged_by_at_least_two"),
)

method_comparison



## 22. Tabela najbardziej nietypowych obiektów

Do tabeli wynikowej dołączamy identyfikator lub nazwę obiektu, jeżeli są dostępne.


In [ ]:

all_asteroids_df = asteroids_lf.collect()

identifier_candidates = [
    "neo_reference_id",
    "name",
    "id",
]

identifier_columns = [
    column
    for column in identifier_candidates
    if column in all_asteroids_df.columns
]

result_base_columns = [
    *identifier_columns,
    *selected_features,
]

asteroid_anomaly_results = (
    all_asteroids_df
    .select(result_base_columns)
    .hstack(anomaly_flags)
)

asteroid_anomaly_results.sort(
    [
        "number_of_methods_flagging",
        "isolation_score",
    ],
    descending=[True, True],
).head(20)



## 23. Wizualizacja anomalii w projekcji PCA

PCA służy tu wyłącznie do wizualizacji. Detektor Isolation Forest działa na pełnej, przeskalowanej przestrzeni cech, a nie na samych dwóch komponentach.


In [ ]:

anomaly_pca = PCA(n_components=2)
X_anomaly_pca = anomaly_pca.fit_transform(
    X_anomaly
)

plt.figure(figsize=(8.5, 6))
scatter = plt.scatter(
    X_anomaly_pca[:, 0],
    X_anomaly_pca[:, 1],
    c=(isolation_labels == -1).astype(int),
    s=20,
    alpha=0.65,
)
plt.title("Isolation Forest — anomalie w projekcji PCA")
plt.xlabel(
    f"PC1 ({anomaly_pca.explained_variance_ratio_[0]:.1%} wariancji)"
)
plt.ylabel(
    f"PC2 ({anomaly_pca.explained_variance_ratio_[1]:.1%} wariancji)"
)
plt.legend(
    *scatter.legend_elements(),
    title="0 = typowy, 1 = anomalia",
)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



## 24. Zapis wyników

Tabela może zostać przekazana ekspertowi dziedzinowemu do ręcznej weryfikacji albo wykorzystana w aplikacji Streamlit.


In [ ]:

ANOMALY_OUTPUT_PATH = (
    OUTPUT_DIR / "nasa_asteroids_anomaly_results.parquet"
)

asteroid_anomaly_results.write_parquet(
    ANOMALY_OUTPUT_PATH
)

print(f"Zapisano: {ANOMALY_OUTPUT_PATH}")



## Podsumowanie części 2

### Brakujące dane

```text
dane z brakami
→ fit imputera na train
→ transform train i test
→ kompletna macierz
```

- `SimpleImputer` jest prosty, szybki i często stanowi dobry baseline.
- `KNNImputer` wykorzystuje podobne rekordy, ale zależy od skali i doboru sąsiadów.
- `IterativeImputer` modeluje każdą niekompletną cechę na podstawie pozostałych.
- Brak wartości nie jest tym samym co zero.
- Wskaźnik braku może czasem sam stanowić wartościową cechę.

### Anomaly detection

```text
kompletne, przeskalowane cechy
→ kilka definicji nietypowości
→ ranking rekordów
→ ręczna interpretacja
```

Algorytm wskazuje rekordy odbiegające od wzorca. Nie rozstrzyga, czy są one:

- błędem,
- nadużyciem,
- rzadkim, ale poprawnym przypadkiem,
- nowym zjawiskiem,
- wartościową obserwacją wymagającą uwagi.



---

## Część 3. Klasyfikacja i podejmowanie decyzji

Problem analityczny:

> Czy na podstawie parametrów fizycznych i orbitalnych można zaklasyfikować asteroidę jako potencjalnie niebezpieczną?

Przepływ:

```text
dane i target
→ podział train/test
→ ColumnTransformer
   ├── cechy numeryczne: KNNImputer → StandardScaler
   └── cechy kategoryczne: SimpleImputer → OneHotEncoder
→ Pipeline
→ klasyfikator
→ predict / predict_proba
→ confusion matrix i metryki
→ próg decyzji
```

Porównamy:

- model bazowy `DummyClassifier`,
- `LogisticRegression`,
- `RandomForestClassifier`.

> `Hazardous` oznacza klasę potentially hazardous asteroid, a nie prognozę konkretnego zderzenia z Ziemią.



## 25. Importy do klasyfikacji


In [ ]:

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Pandas:", pd.__version__)



## 26. Przygotowanie zmiennej docelowej

W klasyfikacji potrzebujemy etykiety, którą model ma przewidywać. Nazywamy ją:

- targetem,
- zmienną docelową,
- etykietą klasy.

W popularnym zbiorze NASA Asteroids kolumna nazywa się zwykle `Hazardous`. Po ujednoliceniu nazw oczekujemy `hazardous`.


In [ ]:

classification_df = asteroids_lf.collect().to_pandas()

TARGET_CANDIDATES = [
    "hazardous",
    "is_hazardous",
    "potentially_hazardous",
]

target_column = next(
    (
        column
        for column in TARGET_CANDIDATES
        if column in classification_df.columns
    ),
    None,
)

if target_column is None:
    raise ValueError(
        "Nie znaleziono targetu Hazardous. "
        f"Dostępne kolumny: {classification_df.columns.tolist()}"
    )


def normalize_binary_target(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        unique = set(numeric.dropna().unique().tolist())

        if unique.issubset({0, 1}):
            return numeric.astype("Int64")

        return (numeric > 0).astype("Int64")

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": 1,
        "yes": 1,
        "y": 1,
        "1": 1,
        "hazardous": 1,
        "false": 0,
        "no": 0,
        "n": 0,
        "0": 0,
        "not hazardous": 0,
        "non-hazardous": 0,
    }

    return normalized.map(mapping).astype("Int64")


classification_df["target"] = normalize_binary_target(
    classification_df[target_column]
)

classification_df = classification_df.dropna(
    subset=["target"]
).copy()

classification_df["target"] = (
    classification_df["target"].astype(int)
)

target_distribution = (
    classification_df["target"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="records")
    .sort_values("class")
)

target_distribution["share"] = (
    target_distribution["records"]
    / target_distribution["records"].sum()
)

target_distribution



### Niezbalansowane klasy

W zbiorach dotyczących ryzyka klasa pozytywna jest często rzadsza.

Dlatego sama accuracy może być myląca. Model przewidujący zawsze klasę większościową może uzyskać wysoką accuracy, ale nie wykryć ani jednego potencjalnie niebezpiecznego obiektu.

Będziemy analizować także:

- precision,
- recall,
- F1,
- balanced accuracy,
- ROC AUC,
- average precision.



## 27. Cechy numeryczne i kategoryczne

Zbiór NASA jest głównie numeryczny. `ColumnTransformer` pokażemy jednak w wersji obsługującej oba typy danych.

Najpierw szukamy użytecznej kolumny kategorycznej. Jeżeli jej nie ma, tworzymy interpretowalną kategorię `inclination_band` na podstawie nachylenia orbity i usuwamy źródłową kolumnę `inclination` z cech numerycznych, aby nie przekazywać tej samej informacji dwa razy.


In [ ]:

numeric_features = [
    feature
    for feature in selected_features
    if feature in classification_df.columns
]

categorical_candidates = [
    "orbiting_body",
    "equinox",
    "orbit_class_type",
    "orbit_class_description",
]

categorical_features = [
    column
    for column in categorical_candidates
    if (
        column in classification_df.columns
        and classification_df[column].nunique(dropna=True) > 1
        and classification_df[column].nunique(dropna=True) <= 30
    )
]

engineered_category = None

if not categorical_features:
    if "inclination" in classification_df.columns:
        engineered_category = "inclination_band"
        classification_df[engineered_category] = pd.cut(
            pd.to_numeric(
                classification_df["inclination"],
                errors="coerce",
            ),
            bins=[-np.inf, 10, 25, np.inf],
            labels=["low", "medium", "high"],
        )
        categorical_features = [engineered_category]
        numeric_features = [
            feature
            for feature in numeric_features
            if feature != "inclination"
        ]

    elif "eccentricity" in classification_df.columns:
        engineered_category = "eccentricity_band"
        classification_df[engineered_category] = pd.cut(
            pd.to_numeric(
                classification_df["eccentricity"],
                errors="coerce",
            ),
            bins=[-np.inf, 0.3, 0.6, np.inf],
            labels=["low", "medium", "high"],
        )
        categorical_features = [engineered_category]
        numeric_features = [
            feature
            for feature in numeric_features
            if feature != "eccentricity"
        ]

if not numeric_features:
    raise ValueError("Nie znaleziono cech numerycznych do modelu.")

print("Cechy numeryczne:")
for feature in numeric_features:
    print("-", feature)

print("\nCechy kategoryczne:")
for feature in categorical_features:
    print("-", feature)

if engineered_category:
    print(
        "\nUtworzono demonstracyjną cechę:",
        engineered_category,
    )



## 28. Kontrola potencjalnego target leakage

NASA definiuje potentially hazardous asteroid między innymi przez:

- `absolute_magnitude`,
- `minimum_orbit_intersection`, czyli MOID.

Jeżeli target został utworzony bezpośrednio z tych samych cech, model może po prostu odtworzyć znaną regułę.

Nie musi to być błędem, gdy celem jest automatyczne odtwarzanie tej klasyfikacji. Jest jednak problemem, gdy przedstawiamy wynik jako odkrycie nowej zależności albo przewidywanie na podstawie niezależnych informacji.

Przygotujemy dwa warianty:

1. **all features** — zawiera również cechy definicyjne,
2. **without definition features** — usuwa cechy bezpośrednio definiujące target.


In [ ]:

TARGET_DEFINITION_FEATURES = [
    feature
    for feature in [
        "absolute_magnitude",
        "minimum_orbit_intersection",
    ]
    if feature in numeric_features
]

numeric_features_without_definition = [
    feature
    for feature in numeric_features
    if feature not in TARGET_DEFINITION_FEATURES
]

print(
    "Cechy potencjalnie odtwarzające definicję targetu:",
    TARGET_DEFINITION_FEATURES,
)
print(
    "Liczba cech numerycznych po ich usunięciu:",
    len(numeric_features_without_definition),
)



## 29. Podział na dane treningowe i testowe

Zbiór treningowy służy do:

- obliczenia parametrów imputacji,
- dopasowania skalera,
- poznania kategorii dla OneHotEncoder,
- wyuczenia modelu.

Zbiór testowy pozostaje niewidoczny podczas uczenia i służy do końcowej oceny.

Używamy `stratify=y`, aby zachować podobny udział klasy pozytywnej w obu częściach.


In [ ]:

all_input_features = [
    *numeric_features,
    *categorical_features,
]

X = classification_df[all_input_features].copy()
y = classification_df["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "dataset": ["train", "test"],
        "records": [len(y_train), len(y_test)],
        "positive_share": [
            y_train.mean(),
            y_test.mean(),
        ],
    }
)

split_summary



## 30. ColumnTransformer

`ColumnTransformer` stosuje różne operacje do różnych grup kolumn.

### Cechy numeryczne

```text
KNNImputer
→ StandardScaler
```

### Cechy kategoryczne

```text
SimpleImputer(strategy="most_frequent")
→ OneHotEncoder(handle_unknown="ignore")
```

W tym przykładzie zachowujemy kolejność KNNImputer → StandardScaler, aby pokazać typowy, czytelny pipeline. Trzeba jednak pamiętać, że KNNImputer korzysta z odległości, więc przy bardzo różnych jednostkach warto rozważyć wcześniejszą normalizację lub bardziej wyspecjalizowany preprocessing.


In [ ]:

def build_preprocessor(
    numeric_columns: list[str],
    categorical_columns: list[str],
) -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                KNNImputer(
                    n_neighbors=5,
                    weights="distance",
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent",
                ),
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    transformers = [
        (
            "numeric",
            numeric_pipeline,
            numeric_columns,
        ),
    ]

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor(
    numeric_features,
    categorical_features,
)

preprocessor



## 31. Model bazowy

Zanim ocenimy właściwe modele, tworzymy baseline.

`DummyClassifier(strategy="prior")` ignoruje cechy i przewiduje klasy zgodnie z rozkładem klas w danych treningowych.

Model ML powinien dostarczyć wyraźnie lepszy wynik niż taki prosty punkt odniesienia.


In [ ]:

baseline_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="prior",
                random_state=42,
            ),
        ),
    ]
)

baseline_pipeline.fit(X_train, y_train)



## 32. Logistic Regression i Random Forest

### Logistic Regression

- model liniowy,
- szybki i stosunkowo łatwy do interpretacji,
- zwraca prawdopodobieństwo klasy,
- korzysta ze skalowania,
- może być dobrym baseline'em predykcyjnym.

### Random Forest

- zespół wielu drzew decyzyjnych,
- potrafi modelować nieliniowe zależności i interakcje,
- zwykle nie wymaga skalowania, ale może korzystać ze wspólnego pipeline'u,
- również udostępnia `predict_proba`.

Ustawiamy `class_weight="balanced"`, aby zwiększyć wagę rzadszej klasy pozytywnej.


In [ ]:

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2_000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            build_preprocessor(
                numeric_features,
                categorical_features,
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

logistic_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

print("Modele zostały dopasowane.")



## 33. Funkcja oceny modeli

`predict()` zwraca decyzję klasową.

`predict_proba()` zwraca prawdopodobieństwo klasy pozytywnej. Pozwala ono:

- tworzyć ranking obiektów,
- zmieniać próg decyzji,
- dostosować model do kosztu różnych błędów.

Prawdopodobieństwa modelu nie zawsze są idealnie skalibrowane. W systemie wysokiego ryzyka należałoby dodatkowo ocenić kalibrację.


In [ ]:

def evaluate_classifier(
    name: str,
    model: Pipeline,
    X_eval: pd.DataFrame,
    y_eval: pd.Series,
) -> dict[str, float | str]:
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(
            y_eval,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_eval,
            predictions,
        ),
        "precision": precision_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_eval,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_eval,
            probabilities,
        ),
        "average_precision": average_precision_score(
            y_eval,
            probabilities,
        ),
    }


models = {
    "DummyClassifier": baseline_pipeline,
    "LogisticRegression": logistic_pipeline,
    "RandomForest": random_forest_pipeline,
}

evaluation_results = pd.DataFrame(
    [
        evaluate_classifier(
            name,
            model,
            X_test,
            y_test,
        )
        for name, model in models.items()
    ]
).sort_values(
    "average_precision",
    ascending=False,
)

evaluation_results



## 34. Confusion matrix

Dla klasy pozytywnej:

- **TP** — model poprawnie wykrył potentially hazardous,
- **FN** — obiekt pozytywny został przeoczony,
- **FP** — obiekt bez tej etykiety został oznaczony jako pozytywny,
- **TN** — poprawnie rozpoznano klasę negatywną.

W zadaniu dotyczącym ryzyka koszt FN może być znacznie większy niż koszt FP. Wtedy recall może być ważniejsze niż sama accuracy.


In [ ]:

selected_model_name = "LogisticRegression"
selected_model = models[selected_model_name]

test_predictions = selected_model.predict(X_test)
test_probabilities = selected_model.predict_proba(
    X_test
)[:, 1]

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "not hazardous",
            "potentially hazardous",
        ],
        zero_division=0,
    )
)


In [ ]:

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=[
        "not hazardous",
        "potentially hazardous",
    ],
    cmap=None,
    values_format="d",
)
plt.title(
    f"Confusion matrix — {selected_model_name}"
)
plt.tight_layout()
plt.show()



## 35. Prawdopodobieństwo i próg decyzji

Domyślnie `predict()` zwykle wykorzystuje próg 0,5:

```text
p >= 0.5 → klasa pozytywna
p < 0.5  → klasa negatywna
```

To nie jest prawo natury. Możemy obniżyć próg, jeżeli ważniejsze jest wykrycie większej liczby przypadków pozytywnych, akceptując więcej fałszywych alarmów.


In [ ]:

def metrics_for_threshold(
    y_true: pd.Series,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    predictions = (
        probabilities >= threshold
    ).astype(int)

    return {
        "threshold": threshold,
        "predicted_positive_share": predictions.mean(),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
    }


threshold_results = pd.DataFrame(
    [
        metrics_for_threshold(
            y_test,
            test_probabilities,
            threshold,
        )
        for threshold in [
            0.20,
            0.30,
            0.40,
            0.50,
            0.60,
            0.70,
            0.80,
        ]
    ]
)

threshold_results


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    marker="o",
    label="precision",
)
plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    marker="o",
    label="recall",
)
plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    marker="o",
    label="f1",
)
plt.title("Wpływ progu decyzji na metryki")
plt.xlabel("Próg")
plt.ylabel("Wartość metryki")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



## 36. Przykładowe predykcje

Wynik probabilistyczny jest często użyteczniejszy niż sama etykieta.

Możemy:

- sortować obiekty według wyniku,
- kierować przypadki wysokiego ryzyka do pilnej weryfikacji,
- pozostawiać przypadki niepewne ekspertowi,
- automatyzować tylko wyniki o dużej pewności.


In [ ]:

prediction_preview = X_test.copy()
prediction_preview["actual_target"] = y_test.to_numpy()
prediction_preview["predicted_probability"] = (
    test_probabilities
)
prediction_preview["predicted_class_0_5"] = (
    test_probabilities >= 0.5
).astype(int)

prediction_preview.sort_values(
    "predicted_probability",
    ascending=False,
).head(15)



## 37. Demonstracja cech bezpośrednio definiujących target

Budujemy drugi model Logistic Regression bez:

- `absolute_magnitude`,
- `minimum_orbit_intersection`.

Porównanie nie ma dowodzić, że tych cech nigdy nie wolno używać. Pokazuje, że cel modelu musi być jasno zdefiniowany:

- automatyczne odtworzenie istniejącej reguły,
- czy przewidywanie etykiety na podstawie niezależnych informacji?


In [ ]:

if (
    TARGET_DEFINITION_FEATURES
    and numeric_features_without_definition
):
    reduced_input_features = [
        *numeric_features_without_definition,
        *categorical_features,
    ]

    X_reduced = classification_df[
        reduced_input_features
    ].copy()

    X_train_reduced = X_reduced.loc[
        X_train.index
    ]
    X_test_reduced = X_reduced.loc[
        X_test.index
    ]

    reduced_logistic_pipeline = Pipeline(
        steps=[
            (
                "preprocessing",
                build_preprocessor(
                    numeric_features_without_definition,
                    categorical_features,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2_000,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )

    reduced_logistic_pipeline.fit(
        X_train_reduced,
        y_train,
    )

    leakage_comparison = pd.DataFrame(
        [
            evaluate_classifier(
                "Logistic — wszystkie cechy",
                logistic_pipeline,
                X_test,
                y_test,
            ),
            evaluate_classifier(
                "Logistic — bez cech definicyjnych",
                reduced_logistic_pipeline,
                X_test_reduced,
                y_test,
            ),
        ]
    )

    leakage_comparison
else:
    print(
        "W zbiorze nie ma obu cech definicyjnych "
        "lub po ich usunięciu nie zostały cechy numeryczne."
    )



## 38. Inne rodzaje data leakage

### Preprocessing na całym zbiorze

Błędnie:

```python
X_imputed = imputer.fit_transform(X)
X_train, X_test = train_test_split(X_imputed)
```

Poprawnie:

```text
train/test split
→ pipeline.fit(train)
→ pipeline.predict(test)
```

### Informacje z przyszłości

Przewidywanie wyniku na dzień decyzji z użyciem kolumny utworzonej już po podjęciu decyzji.

Przykłady:

- końcowy status sprawy,
- data zamknięcia procesu,
- wartość znana dopiero po zdarzeniu,
- agregaty zawierające przyszłe transakcje.

### Identyfikatory i techniczne artefakty

Identyfikator, kolejność importu albo nazwa pliku mogą przypadkowo kodować klasę, mimo że nie opisują analizowanego zjawiska.



## 39. Opcjonalnie: Gradient Boosting

Dodatkowym modelem może być `HistGradientBoostingClassifier` albo `GradientBoostingClassifier`.

Modele boostingowe często uzyskują dobre wyniki na danych tabelarycznych, ale:

- mają kolejne hiperparametry,
- wymagają osobnego strojenia,
- nie zmieniają głównej lekcji tej części.

Dlatego w podstawowym demo pozostajemy przy Logistic Regression i Random Forest.



## 40. Zapis finalnego zestawienia predykcji

Na potrzeby aplikacji PoC zapisujemy wyniki zbioru testowego razem z prawdopodobieństwem modelu.


In [ ]:

CLASSIFICATION_OUTPUT_PATH = (
    OUTPUT_DIR
    / "nasa_asteroids_classification_predictions.parquet"
)

prediction_output = pl.from_pandas(
    prediction_preview.reset_index(
        names="source_index"
    )
)

prediction_output.write_parquet(
    CLASSIFICATION_OUTPUT_PATH
)

print(f"Zapisano: {CLASSIFICATION_OUTPUT_PATH}")



## Podsumowanie części 3

### Klasyfikacja

```text
cechy → model → klasa
```

Przykład: potentially hazardous / not hazardous.

### Regresja

```text
cechy → model → wartość liczbowa
```

Przykład: przewidywana odległość minięcia, czas, cena lub wielkość sprzedaży.

### Kompletny model analityczny

```text
ColumnTransformer
→ imputacja
→ skalowanie
→ kodowanie kategorii
→ klasyfikator
```

### Najważniejsze lekcje

1. Model porównujemy z prostym baseline'em.
2. Dane testowe pozostają niewidoczne podczas uczenia.
3. Accuracy nie wystarcza przy niezbalansowanych klasach.
4. `predict_proba` pozwala dostosować próg decyzji.
5. Pipeline obejmuje preprocessing i model.
6. Bardzo wysoki wynik może być skutkiem target leakage albo odtwarzania definicji etykiety.
7. Model wspiera decyzję, ale koszt błędów i próg działania muszą wynikać z kontekstu.



## 41. Zapis pipeline'u dla aplikacji Streamlit

Zapisujemy kompletny pipeline Logistic Regression, zawierający
`ColumnTransformer`, imputację, skalowanie, kodowanie kategorii
oraz klasyfikator.

Aplikacja Streamlit może dzięki temu przyjmować nowe rekordy
w takim samym schemacie jak dane treningowe.


In [ ]:

import joblib

STREAMLIT_MODEL_PATH = (
    OUTPUT_DIR
    / "nasa_asteroids_logistic_pipeline.joblib"
)

joblib.dump(
    logistic_pipeline,
    STREAMLIT_MODEL_PATH,
)

print(f"Zapisano pipeline: {STREAMLIT_MODEL_PATH}")
